# pymarc extended
This notebook goes through some useful functions developed during various metadata projects. Most of the functions use basic pymarc methods to codify operations that are often performed when dealing with analysis and bulk updating of Marc records.

Get access to the functions by cloning the tkpy Git repository - easily done by clicking on the link below: 

[Clone olufine/tkpy](https://k8s.nb.no/jupyter/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Folufine%2Ftkpy&urlpath=lab%2Ftree%2Ftkpy%2FREADME.md&branch=main) 

This should result in a **tkpy** folder in the left pane.

**Note:** The tkpy repository contains many more functions than those mentioned below. In this notebook we'll focus on the ones used most often and therefore deemed most useful. 

Please feel free to try out others also!



In [3]:
import pymarc
import re  #Regular expression handling
import os
import sys
import csv
repopath=os.path.abspath('/home/jovyan/tkpy')
if repopath not in sys.path:
    sys.path.append(repopath)
import marcpy1, marcpy2, iomarc, iogeneral, utils, utilmarc 
from marcpy1 import valueCounter, fieldCounter, subfieldCounter2
from marcpy2 import selectAssigned, selectMissingFields, filterRecordsByControlField, filterRecordsByLeader, filterRecords, indexRecords, indexRecords2, indexRecords3
from iomarc import printFields, writeMarcToFile, makeRows4FieldSpec, colNames 
from iogeneral import writeToFile, writeRowsToCSV, readCsvRows
from utils import oneLineStr, modifyString, fillz
from utilmarc import replaceControlField, replaceLeader, normalizedISBN, isbnConvert10to13 

## 1. Counting fields and subfields in your dataset
* **fieldCounter(records)** : Returns a dict with fieldtags as keys, and the number of occurernces as values
* **subfieldCounter2(records, delimiter='£')** : Returns a dict with the strings <fieldtag>£<subfieldtag> (e.g. '100£a') as keys, and numnber of occurrences as values

## 2. Filtering records - extracting subsets based on various criteria


### 2.1. Filtering records based on whether or not a certain field is present or missing
* **selectAssigned(records, fieldtag, subfields=None,  compReq='all')**
* **selectMissingFields(records, fieldtags, all=True)**

Examples: 
```
# find the records in myrecords containing at least one access point for work or expression
selectAssigned(myrecords, '700', subfields=['t'])
# find the records in myrecords that are part of at least one bibliography or virtual collection
selectAssigned(myrecords, '913')
# find the records in myrecords in which neither 260 nor 264 exist
selectMissingFields(myrecords, ['260', '264'])

### 2.2. Filtering records based on field values
* **filterRecords(records, regpattern, fieldtags, subfieldtags=[])**
* **filterRecordsByControlField(records, regpattern, fieldtag, posint)**
* **filterRecordsByLeader(records, regpattern, posint=(0,24))**

Examples:
```
# find the records in myrecords which are assigned genres from ntsf
filterRecords(myrecords, 'ntsf', ['655'], subfieldtags=['2'])
# find the records in myrecords with original titles i 246 (whether the relationship term is capitalized or not)
filterRecords(myrecords, '(O|o)riginaltittel', ['246'], subfieldtags=['i'])
# find the records in myrecords with ddc starting with 948 or with 948 behind the dot
filterRecords(myrecords, '^(948)|(\\d{3}\\.948)', ['082'], subfieldtags=['a'])
# find the records in myrecords published in the 20th century
filterRecordsByControlField(myrecords, '19', '008', (7,9))
# find the records in myrecords which represent music resources (notated, manuscript or recorded music)
filterRecordsByLeader(myrecords, 'c|d|j', posint=(6,7))
```
**Note:** the *regpattern* argument in the filtering functions above must be regular expression. Here is the [documentation of Python's re module](https://docs.python.org/3/library/re.html) \
Regular expressions are **text patterns** to match against textual data. Their complexity can vary from plain strings (like 'ntsf') to very complex patterns. \
However, for our use only a few special characters are necessary, see for example this [Quick Reference](https://blog.hpc.qmul.ac.uk/Beginners-Guide-to-Regular-Expressions/#quick-reference)

What if we need the **complement** of the records found by filtering (those that was **not** included in the list of records returnd by e.g. filterRecords? 
One way of solving that is to use set operations. 
```
# Give me a list of the records with genres from ntsf
w655ntsf=filterRecords(myrecords, 'ntsf', ['655'], subfieldtags=['2'])
complSet=set(myrecords).difference(set(w655ntsf))
without655ntsf=list(complSet)
```

## 3. Extracting and counting all distinct values in a specific field or part of a positional field
* **valueCounter(records, fieldtags, subfieldtags=None, fldPart=None, slice=None, separateCounting=False, delimiter='£', leadertag='000', countDupl=True)**
Examples:
````
# Which vocabularies are used as subjects in my dataset? (the distribution of the various source codes in 650, as a dict)
valueCounter(myRecords, ['650'], subfieldtags=['2']) 
# How are the various bibliographic levels distributed in my dataset?
valueCounter (myRecords, ['000'], slice=(7,8))
# In which countries are the resources in my dataset published? (the distribution of publication countries as encoded in 008, as a dict)
valueCounter (myRecords, ['008'], slice=(15,18))
````

## 4. Indexing your dataset
* **indexRecords(records)**: Indexing on record id (MMS Id)
* **indexRecords2(records, fieldtag, subfieldtags=None, sep='£', norm=None)**: Indexing on one field  and specific subfieds value
* **indexRecords4(records, keySpec, sep='$')**: Indexing on multiple fields and subfields, as specified in keySpec

Examples:
```
# index my dataset on (generate a dict with MMS Id as keys, and its corresponding record as value)
indexRecords(myrecords) 
# index my dataset on creator names (generate a dict with value of field 100 subfield a as key, and the list of matching records as value)
indexRecords2(myrecords, '100', subfieldtags['a'], sep='$', norm='lower')
# index my dataset on creator name + title (generate a dict with author+title as keys, and the list of matching records as value)
indexRecords4(myrecords, {'100':['a', 'd'], '245':['a']}, sep='$') 
```

## 5. Updating records
* **addField(rec, ftag, subspec, inds=(' ', ' '))**
* **replaceControlField(record, fild, newVal)**
* **replaceLeader(record, newLdrVal)**

Examples:

```
# Create a new Field object and add it to the first record of myrecords, but only if 913.a Littforsk b.NB is not already there
addfield(myrecords[0], '913', {'a' : 'Littforsk', 'b' : 'NB'})
# Creates e new 001 field with value '999923031939002201', and remove the old one
replaceControlField(myrecords[0], '001', '999923031939002201')
# replace leader with a new value
replaceLeader(myrecords[0], '02601nam a22003735c 4500')
```

## 6. Displaying records in the notebook
For displaying a single record or field, just call **print** with the Record or Field object as argument
For displaying sspecific fields in a set of records, use the function below:
* **printFields(records, fldtags)**

Example:
```
# Print the value of the specified fields for the first 10 records of myrecords
printFields(myrecords[0:10], ['100', '110', '111', '240', '245', '264, '300', '655', '773'])
```

## 7. Writing to csv file

### 7.1. The basics
Writing a table of rows to a .csv (text) file is easy.
Collect the data you want to write in a list of tuples (or list of lists)

Example:
```
data=[('Hei', 'Og', 'Hå'),('Og', 'Hildrande', 'Du')]
with open('test.csv', 'w', newline='',  encoding = 'utf-8') as f:
    thiswriter=csv.writer(f, delimiter = '|')
    for rw in data:
            thiswriter.writerow(rw)
    f.close()
```

In [2]:
data=[('Hei', 'Og', 'Hå'), ('Og', 'Hildrande', 'Du')]
with open('test.csv', 'w', newline='',  encoding = 'utf-8') as f:
    thiswriter=csv.writer(f, delimiter = '|')
    for rw in data:
            thiswriter.writerow(rw)
    f.close()

### 7.2 Writing Marc records (selected fields) to csv file
Sometimes, there is a need to export information from Marc records to a more human-friendly format. \
For example to **.csv**, which again can be loaded into Excel \
While the basic method above is suitable for simple data, often our starting point are a set og Marc records, from which we need to pick and choose information to export in a flexible yet detailed manner. The main function for this is described below.

* **makeRows4FieldSpec (record, tagSpec, colSpec, rowUnit='001')** : Returns a list of n tuples (rows) representing record as specified by tagSpec. n is the number of occurrences of rowUnit in record. As long as field 001 is used as rowUnit, this function returns a list of 1 tuple. 
* **tagspec** is a dict with fieldtags as keys, and a list of subfieldcodes as values, specifying which fields and subfields should be part of teh export
* **colSpec** is a dict with fieldtags as keys, and True or False as values, specifying whether the subfields should be exported to separate columns or in the same column.

Example:
```
# Specify which fields and subfields to include in the export
expfields={'001':[],'100':[],'110':[], '111':[], '245':['a', 'b', 'c']}   #Export the whole value of 100/110/111, and subfields a,b and c of 245
# Specify how many columns to create
expcolumns={'001':False,'100':False,'110':False,'111':False, '245':True}     # One column each for 001,100/110/111, for 245: subfields in separate columns

# Initiate the export table with the heading row. 
tbl=[colNames(expfields, expfields)]
# Loop through the records to be included in the export, adding a new row to for each one
for rec in myrecords:
    tbl.extend(makeRows4FieldSpec(rec, fieldspec, colspec, rowUnit='001'))

# Now tbl is the complete list of rows to export
# Write tbl to file
writeRowsToCSV('myExportedData.csv', tbl)
```